In [ ]:

pip install qiskit numpy matplotlib ipywidgets

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_bloch_multivector
from IPython.display import display


# ---------------------------------------------------
# Utility Functions
# ---------------------------------------------------

def bloch_coordinates(state):
    """Return Bloch vector components."""
    alpha, beta = state[0], state[1]
    x = 2 * np.real(np.conj(alpha) * beta)
    y = 2 * np.imag(np.conj(alpha) * beta)
    z = np.abs(alpha)**2 - np.abs(beta)**2
    return x, y, z

def format_state(state):
    """Format statevector using |0>, |1> notation with 2-decimal precision."""

    labels = ["|0>", "|1>"] if len(state) == 2 else \
             ["|00>", "|01>", "|10>", "|11>"]

    terms = []
    for amp, label in zip(state, labels):
        if not np.isclose(amp, 0):
            real = np.real(amp)
            imag = np.imag(amp)

            if np.isclose(imag, 0):
                coeff = f"{real:.2f}"
            elif np.isclose(real, 0):
                coeff = f"{imag:.2f}i"
            else:
                coeff = f"{real:.2f}{'+' if imag >= 0 else ''}{imag:.2f}i"

            terms.append(f"{coeff}{label}")

    return " + ".join(terms)



# Dedicated output areas (critical for stability)
plot_output = widgets.Output()
info_output = widgets.Output()


def draw_bloch(state, title=""):
    with plot_output:
        plot_output.clear_output(wait=True)
        fig = plot_bloch_multivector(state)
        if title:
            fig.suptitle(title, fontsize=14)
        display(fig)
        plt.close(fig)


def show_info(state):
    with info_output:
        info_output.clear_output(wait=True)

        if len(state) == 2:
            x, y, z = bloch_coordinates(state)
            print(f"Bloch Vector:")
            print(f"X = {x:.3f}")
            print(f"Y = {y:.3f}")
            print(f"Z = {z:.3f}")

        print("\nStatevector:")
        print(format_state(state))


# ---------------------------------------------------
# Section 1 — Bloch Sphere Explorer
# ---------------------------------------------------

theta = widgets.FloatSlider(min=0, max=np.pi, step=0.01, value=0, description='θ')
phi   = widgets.FloatSlider(min=0, max=2*np.pi, step=0.01, value=0, description='φ')

def update_bloch(change=None):
    qc = QuantumCircuit(1)
    qc.ry(theta.value, 0)
    qc.p(phi.value, 0)

    state = Statevector.from_instruction(qc).data

    draw_bloch(state, "Bloch Sphere State")
    show_info(state)

theta.observe(update_bloch, names='value')
phi.observe(update_bloch, names='value')

bloch_controls = widgets.VBox([theta, phi])


# ---------------------------------------------------
# Section 2 — Gate Playground
# ---------------------------------------------------

gate_sequence = widgets.SelectMultiple(
    options=['Id', 'X', 'Y', 'Z', 'H', 'S', 'T'],
    description='Gates'
)

theta2 = widgets.FloatSlider(min=0, max=np.pi, step=0.01, value=0, description='θ')
phi2   = widgets.FloatSlider(min=0, max=2*np.pi, step=0.01, value=0, description='φ')

def update_gates(change=None):
    qc = QuantumCircuit(1)
    qc.ry(theta2.value, 0)
    qc.p(phi2.value, 0)

    for gate in gate_sequence.value:
        getattr(qc, gate.lower())(0)

    state = Statevector.from_instruction(qc).data

    draw_bloch(state, f"Gates Applied: {', '.join(gate_sequence.value)}")
    show_info(state)

theta2.observe(update_gates, names='value')
phi2.observe(update_gates, names='value')
gate_sequence.observe(update_gates, names='value')

gate_controls = widgets.VBox([theta2, phi2, gate_sequence])


# ---------------------------------------------------
# Section 3 — Arbitrary State Builder
# ---------------------------------------------------

alpha_real = widgets.FloatText(value=1.0, description="Re(α)")
alpha_imag = widgets.FloatText(value=0.0, description="Im(α)")
beta_real  = widgets.FloatText(value=0.0, description="Re(β)")
beta_imag  = widgets.FloatText(value=0.0, description="Im(β)")

def update_arbitrary(change=None):
    alpha = complex(alpha_real.value, alpha_imag.value)
    beta  = complex(beta_real.value, beta_imag.value)

    state = np.array([alpha, beta], dtype=complex)

    norm = np.linalg.norm(state)
    if norm == 0:
        return

    state /= norm

    draw_bloch(state, "Arbitrary Qubit State")
    show_info(state)

for w in [alpha_real, alpha_imag, beta_real, beta_imag]:
    w.observe(update_arbitrary, names='value')

arbitrary_controls = widgets.VBox([
    widgets.Label("Define State |ψ⟩ = α|0⟩ + β|1⟩"),
    alpha_real, alpha_imag,
    beta_real, beta_imag
])


# ---------------------------------------------------
# Section 4 — Two-Qubit Demonstrator (Fixed)
# ---------------------------------------------------

# 1. UI Elements
c00_r = widgets.FloatText(value=1.0, description='Re: |00⟩:')
c01_r = widgets.FloatText(value=0.0, description='Re: |01⟩:')
c10_r = widgets.FloatText(value=0.0, description='Re: |10⟩:')
c11_r = widgets.FloatText(value=0.0, description='Re: |11⟩:')
c00_i = widgets.FloatText(value=0.0, description='Im: |00⟩:')
c01_i = widgets.FloatText(value=0.0, description='Im: |01⟩:')
c10_i = widgets.FloatText(value=0.0, description='Im: |10⟩:')
c11_i = widgets.FloatText(value=0.0, description='Im: |11⟩:')

two_gate = widgets.Dropdown(
    options=['Identity', 'CNOT (0→1)', 'SWAP', 'CZ'],
    description='Gate'
)

coef_inputs = widgets.VBox([
    widgets.HTML("<b>Enter Coefficients (e.g., 1.0 and 1.0 for Bell State):</b>"),
    widgets.HBox([c00_r, c00_i]),
    widgets.HBox([c01_r, c01_i]),
    widgets.HBox([c10_r, c10_i]),
    widgets.HBox([c11_r, c11_i])
])

# 2. Logic
def update_two_complex(change=None):
    # Prepare the input vector
    raw_state = np.array([
        complex(c00_r.value,c00_i.value),
        complex(c01_r.value,c01_i.value),
        complex(c10_r.value,c10_i.value),
        complex(c11_r.value,c11_i.value)
    ], dtype=complex)

    # Normalize
    norm = np.linalg.norm(raw_state)
    if norm == 0:
        with info_output:
            info_output.clear_output()
            print("Error: Total probability is zero.")
        return

    state_norm = raw_state / norm

    # Build Circuit
    qc = QuantumCircuit(2)
    # Initialize both qubits with the normalized vector
    qc.initialize(state_norm, [0, 1])

    # Apply Gate
    if "CNOT" in two_gate.value:
        qc.cx(0, 1)
    elif "SWAP" in two_gate.value:
        qc.swap(0, 1)
    elif "CZ" in two_gate.value:
        qc.cz(0, 1)

    # Calculate Statevector
    # Note: Statevector.from_instruction works directly on the circuit object
    state = Statevector.from_instruction(qc)

    # Update UI
    draw_bloch(state.data, f"Two-Qubit State: {two_gate.value}")
    show_info(state.data)

# 3. Observers (Only link the complex update function)
for widget in [c00_r, c00_i, c01_r, c01_i, c10_r, c10_i, c11_r, c11_i, two_gate]:
    widget.observe(update_two_complex, names='value')

two_controls = widgets.VBox([coef_inputs, two_gate])


# ---------------------------------------------------
# Section Selector (Stable UI)
# ---------------------------------------------------

tabs = widgets.Tab(children=[
    bloch_controls,
    gate_controls,
    arbitrary_controls,
    two_controls
])

titles = [
    "Bloch Sphere",
    "Gate Playground",
    "Arbitrary State",
    "Two-Qubit Gates"
]

for i, t in enumerate(titles):
    tabs.set_title(i, t)


display(tabs, plot_output, info_output)

update_bloch()